In [ ]:
from dotenv import load_dotenv
load_dotenv()

import json
import os
import requests

from omegaconf import OmegaConf
from openai import OpenAI
from pathlib import Path

from beyond_hate.train.prompts import coarse_prompt

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

In [3]:
__file__ = '/Users/nilsherrmann/beyond_hate/beyond_hate/eval/eval_gpt.ipynb'  # Mock __file__ for notebook context
# Config paths
project_root = Path(__file__).parent.parent.parent.resolve()

config_base_path = project_root / 'config/default.yaml'

# Load configurations
cfg = OmegaConf.load(config_base_path)

# Paths
hf_path = project_root / cfg.data.paths.hf

In [4]:
# Load labels into a DataFrame
with open(hf_path / 'test_seen.jsonl', 'r') as f:
    test_data = [json.loads(line) for line in f]

with open(project_root / cfg.data.paths.base / 'images_to_annotate.txt', 'r') as f:
    lines = f.readlines()
    annotated_images = [line.strip() for line in lines]

# Keep only annotated images in the test_unseen split
test_data = [item for item in test_data if item['id'] in annotated_images]

In [5]:
# Load prompts
SYSTEM_TEXT = coarse_prompt['system']
USER_TEXT = coarse_prompt['user']

In [6]:
def convert_to_conversation_gpt(text, img_url, system_prompt, user_prompt):
    """Convert text and image URL into a conversation format for GPT models."""
    conversation = [
        {"role": "system", "content": system_prompt},
        {"role": "user",
         "content": [
             {"type": "text", "text": user_prompt.format(text)},
             {"type": "image_url", "image_url": {"url": img_url}},
         ]
        }
    ]
    return conversation

In [7]:
hf_url = 'https://huggingface.co/datasets/neuralcatcher/hateful_memes/resolve/main'

sample = test_data[0]

label = sample['label']
text = sample['text']
img_url = f'{hf_url}/{sample["img"]}'

# Make a HEAD request to follow redirects and get the final URL
response = requests.head(img_url, allow_redirects=True)
resolved_url = response.url

conversation = convert_to_conversation_gpt(text, resolved_url, SYSTEM_TEXT, USER_TEXT)
conversation

[{'role': 'system',
  'content': 'You are a content moderation assistant. Aid me to label images with text as hateful or neutral. Hateful image are defined as containing a direct or indirect attack on people based on characteristics, including ethnicity, race, nationality, immigration status, religion, caste, sex, gender identity, sexual orientation, and disability or disease.'},
 {'role': 'user',
  'content': [{'type': 'text',
    'text': ' Considering the image and its text: "introducing fidget spinner for women". Is the content of the image and its text hateful or neutral?  Respond only with the word "Hateful" or "Neutral".'},
   {'type': 'image_url',
    'image_url': {'url': 'https://cas-bridge.xethub.hf.co/xet-bridge-us/638824321901766b88075239/a7c4fd0bdf802f9c6720355de3a6296416b206964502c1d8f6f8255ae99f6215?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Content-Sha256=UNSIGNED-PAYLOAD&X-Amz-Credential=cas%2F20251022%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20251022T153400Z&X-Amz-Expire

In [ ]:
response = openai_client.chat.completions.create(
    model="gpt-5-mini",
    messages=conversation
)

In [13]:
response.choices[0].message.content

'Neutral'